# P3 Evaluation — PROF-GRPO checkpoint-275

**Purpose:** Evaluate the PROF-GRPO checkpoint at step 275 against the metrics in our plan:
- pass@k (k = 1, 4, 8, 16, 32) — entropy collapse / diversity
- average@16 — Ye et al. replication target (~39.6% for PROF-GRPO)
- RAC — Reasoning-Answer Consistency (LLM-as-judge, Qwen2.5-7B-Instruct)

**Hardware:** Kaggle T4 (16 GB). Run Sections 1–4 in one session, Section 5 (RAC) in a fresh session.

**Checkpoint:** `prof-grpo-checkpoints/checkpoint-275`  
**Base model:** `unsloth/Qwen2.5-Math-1.5B-Instruct` (LoRA rank 16, trained with PROF-filtered GRPO)

---
### Sections
0. Install dependencies
1. Clone repo & import eval module
2. Load model (merge LoRA → base)
3. Load dataset (MATH-500, optional OlympiadBench)
4. Generate 32 responses / problem → save to disk
5. Score pass@k & average@k from saved responses
6. [FRESH SESSION] Score RAC with Qwen2.5-7B-Instruct
7. Plot results

## Section 0 — Install dependencies
Run once per Kaggle session.

In [ ]:
# Core packages
!pip install -q transformers accelerate peft bitsandbytes datasets
# Tokenizer backends (required for Qwen2.5 tokenizer)
!pip install -q sentencepiece tiktoken
# Math answer verification (used by evaluate.py)
!pip install -q math-verify
# Plotting
!pip install -q matplotlib
print('Done')

## Section 1 — Clone repo & import eval module

The `eval/evaluate.py` module in this repo contains all eval logic. Clone the repo so we can import it.

In [ ]:
import os, sys

REPO_URL    = "https://github.com/mehreen019/probe-prof.git"
REPO_BRANCH = "sumaiya/eval"
REPO_DIR    = "/kaggle/working/probe-prof"

if not os.path.exists(REPO_DIR):
    !git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {REPO_BRANCH}
    !git -C {REPO_DIR} pull origin {REPO_BRANCH}

# Add repo root to path so we can import eval/evaluate.py
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from eval.evaluate import (
    load_merged_model, load_eval_dataset,
    generate_responses, compute_pass_at_k,
    compute_rac_scores, aggregate_results, check_answer,
)
print('Imports OK')

## Section 2 — Configuration

Edit the paths here to match where you uploaded the checkpoint on Kaggle.

In [ ]:
# ============================================================
# EDIT THESE PATHS
# ============================================================

# Path to the checkpoint directory you uploaded to Kaggle.
# Upload the zip (prof-grpo-checkpoints-20260517T042419Z-3-001.zip) as a
# Kaggle Dataset, then set this to the unzipped checkpoint folder.
CHECKPOINT_PATH = "/kaggle/input/prof-grpo-checkpoints/prof-grpo-checkpoints/checkpoint-275"

# Where to save generation outputs (survives across cells; save to /kaggle/working)
OUTPUT_DIR = "/kaggle/working/eval_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Generation config
N_RESPONSES   = 32   # generates enough for pass@1/4/8/16/32
TEMPERATURE   = 0.8  # standard sampling temp for pass@k
MAX_NEW_TOKENS = 1024
BATCH_SIZE    = 2    # reduce to 1 if OOM on T4

# Dataset — start with MATH-500, add olympiadbench if time allows
DATASETS = ["math500"]  # options: "math500", "olympiadbench", "aime"
MAX_SAMPLES = None  # set e.g. 100 for a quick sanity check; None = full dataset

print(f"Checkpoint : {CHECKPOINT_PATH}")
print(f"Output dir : {OUTPUT_DIR}")
print(f"N responses: {N_RESPONSES} per problem")
print(f"Datasets   : {DATASETS}")

## Section 3 — Load model

Merges the LoRA adapter into the base model weights. Uses ~7–8 GB VRAM on T4.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, PeftConfig

BASE_MODEL_ID = "unsloth/Qwen2.5-Math-1.5B-Instruct"

# Always load tokenizer from the base model on HuggingFace —
# the checkpoint's tokenizer.json is incomplete regardless of upload structure.
print(f"[load] tokenizer from: {BASE_MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
print(f"[load] tokenizer OK, vocab size: {tokenizer.vocab_size}")

# Check what's actually at the checkpoint path
import os
print(f"\n[load] checkpoint path: {CHECKPOINT_PATH}")
print(f"[load] contents: {os.listdir(CHECKPOINT_PATH)}")

is_adapter = os.path.exists(os.path.join(CHECKPOINT_PATH, "adapter_config.json"))
print(f"[load] is_adapter: {is_adapter}")

if is_adapter:
    print(f"[load] loading base weights from {BASE_MODEL_ID}...")
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )
    print(f"[load] applying LoRA adapter from {CHECKPOINT_PATH}...")
    model = PeftModel.from_pretrained(base, CHECKPOINT_PATH)
    model = model.merge_and_unload()
else:
    # No adapter_config.json — treat as a fully merged model directory
    print(f"[load] no adapter_config.json found, loading as merged model...")
    model = AutoModelForCausalLM.from_pretrained(
        CHECKPOINT_PATH,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )

model.eval()
print(f"\n[load] done. VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"[load] model device: {next(model.parameters()).device}")

## Section 4 — Generate responses

Generates 32 responses per problem and saves to `{OUTPUT_DIR}/{dataset}_responses.jsonl`.  
If the kernel dies mid-way, rerun this cell — it resumes from where it stopped.

In [ ]:
all_results = {}

for dataset_name in DATASETS:
    print(f"\n{'='*55}")
    print(f"  Dataset: {dataset_name}")
    print(f"{'='*55}")

    records = load_eval_dataset(dataset_name, max_samples=MAX_SAMPLES)
    save_path = f"{OUTPUT_DIR}/{dataset_name}_responses.jsonl"

    results = generate_responses(
        model, tokenizer,
        records=records,
        n=N_RESPONSES,
        temperature=TEMPERATURE,
        max_new_tokens=MAX_NEW_TOKENS,
        batch_size=BATCH_SIZE,
        save_path=save_path,
    )

    all_results[dataset_name] = results
    print(f"Saved to: {save_path}")

## Section 5 — Score pass@k and average@k

No generation needed here — works from the saved JSONL.  
This cell can be rerun without re-generating.

In [ ]:
import json

pass_k_results = {}

for dataset_name in DATASETS:
    save_path = f"{OUTPUT_DIR}/{dataset_name}_responses.jsonl"

    # Load saved responses
    results = []
    with open(save_path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                results.append(json.loads(line))

    print(f"\n[{dataset_name}] Scoring {len(results)} problems × {N_RESPONSES} responses...")

    # Check answers (adds 'correct' list to each record in-place)
    for rec in results:
        if "correct" not in rec:
            rec["correct"] = [check_answer(r, rec["answer"]) for r in rec["responses"]]

    # Compute metrics
    ks = tuple(k for k in [1, 4, 8, 16, 32] if k <= N_RESPONSES)
    metrics = compute_pass_at_k(results, ks=ks)
    pass_k_results[dataset_name] = metrics

    # Print
    print(f"\n  {'Metric':<16} Value")
    print(f"  {'-'*28}")
    for k_name in [f"average@{k}" for k in ks] + [f"pass@{k}" for k in ks]:
        if k_name in metrics:
            print(f"  {k_name:<16} {metrics[k_name]*100:.2f}%")

    # Ye et al. replication check (average@16 target: ~39.6%)
    if "average@16" in metrics:
        target = 0.396
        val = metrics["average@16"]
        diff = val - target
        sign = "+" if diff >= 0 else ""
        print(f"\n  Ye et al. target (average@16): {target*100:.1f}%")
        print(f"  Our result:                    {val*100:.2f}%  ({sign}{diff*100:.2f}pp)")

# Save metrics
metrics_path = f"{OUTPUT_DIR}/pass_k_metrics.json"
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(pass_k_results, f, indent=2)
print(f"\nMetrics saved to: {metrics_path}")

## Section 6 — RAC scoring

> **IMPORTANT:** Start a **fresh Kaggle session** before running this section.
> The policy model from Section 3 takes ~7 GB VRAM. The RAC judge (Qwen2.5-7B-Instruct)
> also takes ~7 GB. They cannot both fit on a single T4.
>
> The generation outputs are already saved to disk — this section loads from disk only.

Steps to run RAC:
1. Stop / restart the kernel (Factory reset session)
2. Re-run Section 0 (install deps) and Section 1 (clone repo / imports)
3. Run this section

In [ ]:
# Re-import if running in a fresh session
import os, sys, json

REPO_URL    = "https://github.com/mehreen019/probe-prof.git"
REPO_BRANCH = "sumaiya/eval"
REPO_DIR    = "/kaggle/working/probe-prof"

if not os.path.exists(REPO_DIR):
    !git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {REPO_BRANCH}
    !git -C {REPO_DIR} pull origin {REPO_BRANCH}

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from eval.evaluate import compute_rac_scores, aggregate_results

OUTPUT_DIR = "/kaggle/working/eval_outputs"
DATASETS = ["math500"]

rac_results = {}

for dataset_name in DATASETS:
    print(f"\n{'='*55}")
    print(f"  RAC scoring: {dataset_name}")
    print(f"{'='*55}")

    gen_path = f"{OUTPUT_DIR}/{dataset_name}_responses.jsonl"
    results = []
    with open(gen_path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                results.append(json.loads(line))

    print(f"Loaded {len(results)} problems from {gen_path}")

    rac_save_path = f"{OUTPUT_DIR}/{dataset_name}_rac.jsonl"

    rac_records = compute_rac_scores(
        results,
        judge_model_id="Qwen/Qwen2.5-7B-Instruct",
        correct_only=True,
        max_responses_per_problem=4,
        save_path=rac_save_path,
    )

    rac_results[dataset_name] = rac_records

    import numpy as np
    mean_rac = np.mean([r["mean_rac"] for r in rac_records if r.get("mean_rac") is not None])
    print(f"\n  Mean RAC ({dataset_name}): {mean_rac:.3f}  (n={len(rac_records)} problems)")

print("\nRAC scoring complete.")

## Section 7 — Full results summary & plots

Run after both Section 5 and Section 6 are complete.

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt
import os

OUTPUT_DIR = "/kaggle/working/eval_outputs"
DATASETS = ["math500"]

# Load metrics from disk (works whether you're in the gen session or RAC session)
with open(f"{OUTPUT_DIR}/pass_k_metrics.json") as f:
    pass_k_results = json.load(f)

# ── Pass@k scaling curve ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ks = [1, 4, 8, 16, 32]

for dataset_name in DATASETS:
    metrics = pass_k_results.get(dataset_name, {})

    # pass@k curve
    pass_vals = [metrics.get(f"pass@{k}", None) for k in ks]
    avg_vals  = [metrics.get(f"average@{k}", None) for k in ks]

    valid_ks_pass = [k for k, v in zip(ks, pass_vals) if v is not None]
    valid_pass    = [v for v in pass_vals if v is not None]

    valid_ks_avg  = [k for k, v in zip(ks, avg_vals) if v is not None]
    valid_avg     = [v for v in avg_vals if v is not None]

    ax = axes[0]
    ax.plot(valid_ks_pass, [v * 100 for v in valid_pass], marker="o", label=f"PROF-GRPO ({dataset_name})")
    ax.set_xlabel("k")
    ax.set_ylabel("pass@k (%)")
    ax.set_title("Pass@k Scaling Curve")
    ax.set_xscale("log", base=2)
    ax.set_xticks(ks)
    ax.set_xticklabels(ks)
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax2 = axes[1]
    ax2.plot(valid_ks_avg, [v * 100 for v in valid_avg], marker="s", color="orange", label=f"PROF-GRPO avg@k ({dataset_name})")
    # Ye et al. replication targets (dashed reference lines)
    ax2.axhline(y=39.6, linestyle="--", color="gray", alpha=0.7, label="Ye et al. avg@16 target (39.6%)")
    ax2.axhline(y=37.2, linestyle=":",  color="gray", alpha=0.7, label="Ye et al. baseline (37.2%)")
    ax2.set_xlabel("k")
    ax2.set_ylabel("average@k (%)")
    ax2.set_title("Average@k (Ye et al. Protocol)")
    ax2.set_xscale("log", base=2)
    ax2.set_xticks(ks)
    ax2.set_xticklabels(ks)
    ax2.legend()
    ax2.grid(True, alpha=0.3)

plt.tight_layout()
plot_path = f"{OUTPUT_DIR}/pass_k_curves.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {plot_path}")

In [ ]:
# ── RAC bar chart ────────────────────────────────────────────────────────────
rac_means = {}
for dataset_name in DATASETS:
    rac_path = f"{OUTPUT_DIR}/{dataset_name}_rac.jsonl"
    if not os.path.exists(rac_path):
        print(f"[skip] {rac_path} not found — run Section 6 first")
        continue
    records = []
    with open(rac_path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    valid = [r["mean_rac"] for r in records if r.get("mean_rac") is not None]
    if valid:
        rac_means[dataset_name] = float(np.mean(valid))

if rac_means:
    fig, ax = plt.subplots(figsize=(5, 4))
    labels = list(rac_means.keys())
    vals   = [rac_means[l] for l in labels]
    bars = ax.bar(labels, vals, color=["steelblue"])
    ax.set_ylim(0, 1)
    ax.set_ylabel("Mean RAC score")
    ax.set_title("Reasoning-Answer Consistency (PROF-GRPO)")
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.02, f"{val:.3f}",
                ha="center", fontsize=11)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    rac_plot_path = f"{OUTPUT_DIR}/rac_bar.png"
    plt.savefig(rac_plot_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved to {rac_plot_path}")
else:
    print("No RAC results found. Run Section 6 first.")

In [ ]:
# ── Final summary table ──────────────────────────────────────────────────────
print("\n" + "="*60)
print("  FINAL RESULTS SUMMARY — PROF-GRPO checkpoint-275")
print("="*60)
print(f"  Base model : unsloth/Qwen2.5-Math-1.5B-Instruct")
print(f"  Steps      : 275 / 500  (55% of training)")
print()

for dataset_name in DATASETS:
    metrics = pass_k_results.get(dataset_name, {})
    print(f"  Dataset: {dataset_name}")
    print(f"  {'Metric':<18} {'Value':>8}")
    print(f"  {'-'*28}")
    for k in [1, 4, 8, 16, 32]:
        avg_key  = f"average@{k}"
        pass_key = f"pass@{k}"
        avg_v  = metrics.get(avg_key)
        pass_v = metrics.get(pass_key)
        if avg_v is not None:
            print(f"  {avg_key:<18} {avg_v*100:>7.2f}%")
        if pass_v is not None:
            print(f"  {pass_key:<18} {pass_v*100:>7.2f}%")

    rac_path = f"{OUTPUT_DIR}/{dataset_name}_rac.jsonl"
    if os.path.exists(rac_path):
        records = []
        with open(rac_path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    records.append(json.loads(line))
        valid = [r["mean_rac"] for r in records if r.get("mean_rac") is not None]
        if valid:
            print(f"  {'RAC mean':<18} {np.mean(valid):>8.3f}")
    print()

print(f"  Ye et al. replication targets:")
print(f"    PROF-GRPO average@16 : ~39.6%  (Qwen2.5-Math-1.5B-base)")
print(f"    Baseline GRPO avg@16 : ~37.2%  (Qwen2.5-Math-1.5B-base)")
print(f"  NOTE: Our checkpoint uses -Instruct not -base — direct")
print(f"  comparison is approximate (Instruct typically scores higher)")
print("="*60)